In [1]:
import SimpleITK as sitk

ImportError: DLL load failed while importing _SimpleITK: The specified module could not be found.

In [ ]:
flo_file = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir/wt_M_82Z-10911/coregistered_head.nii.gz"
ref_file = r" /home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/oriented_image.nii.gz"
target = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir/wt_M_82Z-10911/nonlin.nii.gz"
target_transf = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir/wt_M_82Z-10911/transf.hdf5"

In [ ]:
def non_rigid_registration(flo_im: str, ref_im: str, out_transform: str, out_image: str):
    # Reading
    flo = sitk.ReadImage(flo_im)
    ref = sitk.ReadImage(ref_im)

    # Ensure both images have the same data type
    flo = sitk.Cast(flo, ref.GetPixelID())

    # Multi-resolution registration
    regist = sitk.ImageRegistrationMethod()

    # Define a simple callback to monitor registration progress
    def iteration_callback(filter):
        print('\r{0:.5f}'.format(filter.GetMetricValue()), end='')

    # Set the registration parameters
    regist.SetMetricAsMattesMutualInformation()
    regist.SetMetricSamplingStrategy(regist.RANDOM)
    regist.SetMetricSamplingPercentage(0.01)
    regist.SetInterpolator(sitk.sitkLinear)
    regist.SetOptimizerAsGradientDescent(learningRate=0.01, numberOfIterations=100,
                                          maximumStepSizeInPhysicalUnits=0.3)

    regist.AddCommand(sitk.sitkIterationEvent, lambda: iteration_callback(regist))

    # Define multi-resolution settings
    num_levels = 4
    shrink_factors = [4, 2, 1, 1]
    smoothing_sigmas = [2, 1, 0, 0]

    # Perform multi-resolution registration
    transform = None
    for level in range(num_levels):
        # Downsample the images
        flo_down = sitk.Shrink(flo, [shrink_factors[level]] * flo.GetDimension())
        ref_down = sitk.Shrink(ref, [shrink_factors[level]] * ref.GetDimension())

        # Smooth the downsampled images
        flo_smooth = sitk.DiscreteGaussian(flo_down, smoothing_sigmas[level])
        ref_smooth = sitk.DiscreteGaussian(ref_down, smoothing_sigmas[level])

        # Set the initial transform at the current level
        if transform is None:
            transform = sitk.BSplineTransformInitializer(ref_smooth, [10] * ref_smooth.GetDimension(), order=3)

        # Execute registration at the current level
        regist.SetInitialTransform(transform)
        transform = regist.Execute(ref_smooth, flo_smooth)
        regist.SetOptimizerAsConjugateGradientLineSearch()
        # Upsample the transform for the next level
        if level < num_levels - 1:
            transform = sitk.Resample(transform, ref_down, sitk.Transform(), sitk.sitkLinear,
                                      transform.GetPixelIDValue(), transform.GetOrigin(),
                                      transform.GetSpacing(), transform.GetDirection(),
                                      0.0)

    # Save the transform
    sitk.WriteTransform(transform, out_transform)

    # Resample the moving image with the obtained transform
    registered_image = sitk.Resample(flo, ref, transform, sitk.sitkBSpline3, 0.0, ref.GetPixelID())

    # Save the registered image
    sitk.WriteImage(registered_image, out_image, useCompression=True)

non_rigid_registration(flo_im=flo_file,ref_im=ref_file,out_transform=target_transf,out_image=target)